# Lab 6: IP Addressing - Subnetting, Routing, and BGP Prefix Analysis
**CMSC395 - Computer Networks**  
Estimated time: 3-4 hours | Points: 100

---

Work through each cell in order. Do **not** use Python's `ipaddress` module. Build everything using integer arithmetic.

> **Submission:** use **Kernel → Restart Kernel and Run All Cells** before your final push.

## Setup - Imports and Configuration

In [ ]:
# Setup - run this cell first
import math
import struct
import socket
import json
import csv
from pathlib import Path
from collections import defaultdict, Counter

import matplotlib.pyplot as plt
import numpy as np

BGP_FILE    = Path('lab6_bgp.json')
ROUTES_FILE = Path('lab6_routes.csv')

for f in [BGP_FILE, ROUTES_FILE]:
    if not f.exists():
        print(f'WARNING: {f} not found. Re-run nbgitpuller link')
    else:
        print(f'{f}: {f.stat().st_size:,} bytes')

print('Setup complete')

---
## Part 1: IP Addressing Library

Build a Python library. Every component here is imported and used in Parts 2-4.

### Cell 1.1 - Core Integer Operations

Implement the low-level conversion functions for both IPv4 and IPv6.
Do not use `socket.inet_aton` or `socket.inet_ntoa`. Compute from the string directly.
You **may** use `socket.inet_pton` and `socket.inet_ntop` for IPv6 byte conversion only.

In [ ]:
# Cell 1.1 - Core integer operations

# ── IPv4 ──────────────────────────────────────────────────────────

def ip_to_int(ip: str) -> int:
    """
    Convert dotted-decimal IPv4 string to 32-bit integer.
    '192.168.1.1' -> 3232235777
    """
    octets = ip.split('.')
    if len(octets) != 4:
        raise ValueError(f'Invalid IPv4 address: {ip}')
    result = 0
    for o in octets:
        result = (result << 8) | int(o)
    return result


def int_to_ip(n: int) -> str:
    """
    Convert 32-bit integer to dotted-decimal IPv4 string.
    3232235777 -> '192.168.1.1'
    """
    return '.'.join(str((n >> (8 * i)) & 0xFF) for i in (3, 2, 1, 0))


def prefix_to_mask(prefix_len: int, bits: int = 32) -> int:
    """
    Convert prefix length to subnet mask integer.
    prefix_to_mask(24) → 0xFFFFFF00
    Works for both IPv4 (bits=32) and IPv6 (bits=128).
    """
    if prefix_len == 0:
        return 0
    return ((1 << bits) - 1) ^ ((1 << (bits - prefix_len)) - 1)


def mask_to_prefix(mask: int, bits: int = 32) -> int:
    """
    Convert subnet mask integer to prefix length.
    0xFFFFFF00 → 24
    """
    # Count leading ones
    prefix = 0
    for i in range(bits - 1, -1, -1):
        if mask & (1 << i):
            prefix += 1
        else:
            break
    return prefix


def hosts_to_prefix(n_hosts: int, bits: int = 32) -> int:
    """
    Return the smallest prefix length that accommodates n_hosts.
    Accounts for network and broadcast addresses.
    hosts_to_prefix(100) → 25  (supports up to 126 hosts)
    """
    if n_hosts == 0:
        return bits
    host_bits = math.ceil(math.log2(n_hosts + 2))
    return bits - host_bits


# ── IPv6 ──────────────────────────────────────────────────────────

def ip6_to_int(addr: str) -> int:
    """
    Convert IPv6 address string to 128-bit integer.
    May use socket.inet_pton for byte conversion.
    """
    packed = socket.inet_pton(socket.AF_INET6, addr)
    return int.from_bytes(packed, 'big')


def int_to_ip6(n: int) -> str:
    """
    Convert 128-bit integer to IPv6 address string.
    """
    packed = n.to_bytes(16, 'big')
    return socket.inet_ntop(socket.AF_INET6, packed)


# ── Tests ─────────────────────────────────────────────────────────
tests = [
    ('ip_to_int("192.168.1.1")',    ip_to_int('192.168.1.1'),    3232235777),
    ('int_to_ip(3232235777)',        int_to_ip(3232235777),        '192.168.1.1'),
    ('ip_to_int("0.0.0.0")',        ip_to_int('0.0.0.0'),         0),
    ('ip_to_int("255.255.255.255")', ip_to_int('255.255.255.255'), 0xFFFFFFFF),
    ('prefix_to_mask(24)',           prefix_to_mask(24),           0xFFFFFF00),
    ('prefix_to_mask(0)',            prefix_to_mask(0),            0),
    ('prefix_to_mask(32)',           prefix_to_mask(32),           0xFFFFFFFF),
    ('hosts_to_prefix(100)',         hosts_to_prefix(100),         25),
    ('hosts_to_prefix(254)',         hosts_to_prefix(254),         24),
    ('hosts_to_prefix(2)',           hosts_to_prefix(2),           30),
]

all_pass = True
for desc, got, expected in tests:
    ok = (got == expected)
    if not ok:
        all_pass = False
    print(f'  {"OK" if ok else "FAIL"}: {desc} = {got!r} (expected {expected!r})')

print('\nIPv6 conversion tests:')
test_ip6 = '2001:db8::1'
n6 = ip6_to_int(test_ip6)
print(f'  ip6_to_int({test_ip6!r}) = {n6}')
print(f'  int_to_ip6({n6}) = {int_to_ip6(n6)!r}')
assert int_to_ip6(ip6_to_int(test_ip6)) == test_ip6

print('\nAll core tests passed!' if all_pass else '\nSome tests FAILED — fix before continuing')


### Cell 1.2 — Network Class

Implement the `Network` class with all properties and methods.
It must support both IPv4 and IPv6 CIDR notation.

Detect IP version from the presence of `:` in the address string.

In [ ]:
# Cell 1.2 — Network class

# RFC 1918 private ranges (as (network_int, mask_int) tuples)
_PRIVATE_V4 = [
    (ip_to_int('10.0.0.0'),    prefix_to_mask(8)),
    (ip_to_int('172.16.0.0'),  prefix_to_mask(12)),
    (ip_to_int('192.168.0.0'), prefix_to_mask(16)),
]
_PRIVATE_V6_PREFIX = ip6_to_int('fc00::')   # FC00::/7
_PRIVATE_V6_MASK   = prefix_to_mask(7, 128)


class Network:
    """
    Represents an IPv4 or IPv6 network in CIDR notation.
    Supports all standard subnet operations.
    Do not use Python's ipaddress module.
    """

    def __init__(self, cidr: str):
        """
        Construct from CIDR notation.
        Examples: '10.0.0.0/8', '192.168.1.0/24', '2001:db8::/32'
        """
        self._cidr = cidr
        addr_str, prefix_str = cidr.split('/')
        self._prefix_len = int(prefix_str)
        self._is_v6 = ':' in addr_str
        self._bits = 128 if self._is_v6 else 32

        if self._is_v6:
            self._addr_int = ip6_to_int(addr_str)
        else:
            self._addr_int = ip_to_int(addr_str)

        self._mask_int = prefix_to_mask(self._prefix_len, self._bits)
        self._network_int = self._addr_int & self._mask_int

    # ── Internal helpers ──────────────────────────────────────────

    def _int_to_str(self, n: int) -> str:
        return int_to_ip6(n) if self._is_v6 else int_to_ip(n)

    # ── Properties ────────────────────────────────────────────────

    @property
    def network_address(self) -> str:
        return self._int_to_str(self._network_int)

    @property
    def prefix_length(self) -> int:
        return self._prefix_len

    @property
    def netmask(self) -> str:
        return self._int_to_str(self._mask_int)

    @property
    def broadcast_address(self) -> str:
        if self._is_v6:
            return None   # IPv6 has no broadcast
        wildcard = ~self._mask_int & 0xFFFFFFFF
        return int_to_ip(self._network_int | wildcard)

    @property
    def num_hosts(self) -> int:
        host_bits = self._bits - self._prefix_len
        total = 1 << host_bits
        # IPv4: subtract network and broadcast; IPv6: no broadcast
        return max(0, total - 2) if not self._is_v6 else total

    @property
    def host_range(self) -> tuple:
        """
        Return (first_host, last_host) as strings.
        For IPv4: excludes network and broadcast.
        """
        if self._prefix_len >= self._bits - 1:
            return (self.network_address, self.network_address)
        first = self._network_int + (0 if self._is_v6 else 1)
        wildcard = (1 << (self._bits - self._prefix_len)) - 1
        last = (self._network_int | wildcard) - (0 if self._is_v6 else 1)
        return (self._int_to_str(first), self._int_to_str(last))

    @property
    def is_private(self) -> bool:
        if self._is_v6:
            return (self._network_int & _PRIVATE_V6_MASK) == _PRIVATE_V6_PREFIX
        for net, mask in _PRIVATE_V4:
            if (self._network_int & mask) == net:
                return True
        return False

    # ── Containment and overlap ───────────────────────────────────

    def contains(self, ip: str) -> bool:
        """
        Return True if ip is within this network.
        """
        n = ip6_to_int(ip) if self._is_v6 else ip_to_int(ip)
        return (n & self._mask_int) == self._network_int

    def overlaps(self, other: 'Network') -> bool:
        """
        Return True if this network and other share any addresses.
        """
        # Two networks overlap if one contains the other's network address
        # or they share a common prefix
        longer_mask = max(self._mask_int, other._mask_int,
                          key=lambda m: bin(m).count('1'))
        return (self._network_int & longer_mask) == (other._network_int & longer_mask)

    # ── Splitting and expanding ───────────────────────────────────

    def subnets(self, new_prefix: int) -> list:
        """
        Split this network into subnets of new_prefix length.
        new_prefix must be >= self.prefix_length.
        Returns list of Network objects.
        """
        if new_prefix < self._prefix_len:
            raise ValueError(f'new_prefix {new_prefix} must be >= {self._prefix_len}')
        if new_prefix > self._bits:
            raise ValueError(f'new_prefix {new_prefix} exceeds address size {self._bits}')

        step = 1 << (self._bits - new_prefix)
        wildcard = (1 << (self._bits - self._prefix_len)) - 1
        end = self._network_int | wildcard

        result = []
        addr = self._network_int
        while addr <= end:
            result.append(Network(f'{self._int_to_str(addr)}/{new_prefix}'))
            addr += step
        return result

    def supernet(self) -> 'Network':
        """
        Return the next larger network (prefix_length - 1).
        """
        if self._prefix_len == 0:
            raise ValueError('Already at the root prefix')
        new_prefix = self._prefix_len - 1
        new_mask = prefix_to_mask(new_prefix, self._bits)
        new_net = self._network_int & new_mask
        return Network(f'{self._int_to_str(new_net)}/{new_prefix}')

    # ── Class methods ─────────────────────────────────────────────

    @classmethod
    def find_overlaps(cls, prefixes: list) -> list:
        """
        Find all overlapping pairs in a list of CIDR strings.
        Returns list of (Network, Network) tuples.
        """
        networks = [cls(p) for p in prefixes]
        overlaps = []
        for i in range(len(networks)):
            for j in range(i + 1, len(networks)):
                if networks[i].overlaps(networks[j]):
                    overlaps.append((networks[i], networks[j]))
        return overlaps

    @classmethod
    def aggregate(cls, prefixes: list) -> list:
        """
        Reduce a list of CIDR strings to the minimal covering set.
        Returns list of Network objects sorted by network address.
        """
        # Your aggregation algorithm here
        # Hint: sort by (network_int, prefix_len), then repeatedly merge siblings
        pass

    # ── String representation ─────────────────────────────────────

    def __repr__(self):
        return f'Network({self.network_address}/{self._prefix_len})'

    def __str__(self):
        return f'{self.network_address}/{self._prefix_len}'

    def __eq__(self, other):
        return (self._network_int == other._network_int and
                self._prefix_len == other._prefix_len)

    def __lt__(self, other):
        return (self._network_int, self._prefix_len) < (other._network_int, other._prefix_len)


# Quick smoke test
n = Network('192.168.1.0/24')
print(f'Network:    {n}')
print(f'Netmask:    {n.netmask}')
print(f'Broadcast:  {n.broadcast_address}')
print(f'Hosts:      {n.num_hosts}')
print(f'Host range: {n.host_range}')
print(f'Private:    {n.is_private}')
print(f'Contains 192.168.1.1: {n.contains("192.168.1.1")}')
print(f'Contains 192.168.2.1: {n.contains("192.168.2.1")}')
print(f'Supernet: {n.supernet()}')

n6 = Network('2001:db8::/32')
print(f'\nIPv6: {n6}')
print(f'IPv6 private: {n6.is_private}')
print(f'IPv6 hosts: {n6.num_hosts}')


### Cell 1.3 — RoutingTable Class

Implement `RoutingTable` with:
- `add(prefix, next_hop)` — add a route
- `lookup(ip)` — longest-prefix match, return next_hop or None
- `find_covering_routes()` — return pairs where one prefix contains another

In [ ]:
# Cell 1.3 — RoutingTable class

class RoutingTable:
    """
    IPv4 routing table with longest-prefix match lookup.
    """

    def __init__(self):
        self._routes = []   # list of (Network, next_hop, metadata)

    def add(self, prefix: str, next_hop: str, **metadata):
        """
        Add a route to the table.
        metadata: optional keyword args (metric, source, etc.)
        """
        net = Network(prefix)
        self._routes.append((net, next_hop, metadata))

    def lookup(self, ip: str):
        """
        Find the best matching route for ip using longest-prefix match.
        Returns (Network, next_hop, metadata) or None if no match.
        """
        best = None
        best_len = -1

        for net, next_hop, meta in self._routes:
            if net.contains(ip) and net.prefix_length > best_len:
                best = (net, next_hop, meta)
                best_len = net.prefix_length

        return best

    def find_covering_routes(self) -> list:
        """
        Find all pairs (less_specific, more_specific) where one route
        covers another (less_specific contains more_specific's network).
        Returns list of ((Network, next_hop), (Network, next_hop)) tuples.
        """
        pairs = []
        routes = [(net, nh) for net, nh, _ in self._routes]
        for i, (net_i, nh_i) in enumerate(routes):
            for j, (net_j, nh_j) in enumerate(routes):
                if i == j:
                    continue
                # net_i covers net_j if net_i contains net_j's network address
                # and net_i is less specific
                if (net_i.prefix_length < net_j.prefix_length and
                        net_i.contains(net_j.network_address)):
                    pairs.append(((net_i, nh_i), (net_j, nh_j)))
        return pairs

    @property
    def routes(self):
        """Return all routes as list of (Network, next_hop, metadata)."""
        return list(self._routes)

    def __len__(self):
        return len(self._routes)


# Test routing table
rt = RoutingTable()
rt.add('0.0.0.0/0',    'default-gw')
rt.add('10.0.0.0/8',   'internal')
rt.add('10.1.0.0/16',  'datacenter')
rt.add('10.1.2.0/24',  'servers')

test_lookups = [
    ('10.1.2.5',   'servers'),
    ('10.1.3.1',   'datacenter'),
    ('10.2.0.1',   'internal'),
    ('8.8.8.8',    'default-gw'),
]

print('Routing table lookup tests:')
all_pass = True
for ip, expected_nh in test_lookups:
    result = rt.lookup(ip)
    got_nh = result[1] if result else None
    ok = (got_nh == expected_nh)
    if not ok:
        all_pass = False
    matched = f'{result[0]}' if result else 'no match'
    print(f'  {"OK" if ok else "FAIL"}: {ip} → {got_nh} via {matched}')

print('\nCovering routes:')
for (less, nh_less), (more, nh_more) in rt.find_covering_routes():
    print(f'  {less} ({nh_less}) covers {more} ({nh_more})')

print('\nAll lookup tests passed!' if all_pass else '\nSome tests FAILED')


### Cell 1.4 — Library Test Suite

Run a comprehensive test suite covering all `Network` methods and edge cases.
Include at least 2 IPv6 tests. All tests must pass before moving to Part 2.

In [ ]:
# Cell 1.4 — Comprehensive test suite

def test(description, got, expected):
    ok = (got == expected)
    print(f'  {"OK" if ok else "FAIL"}: {description}')
    if not ok:
        print(f'    Expected: {expected!r}')
        print(f'    Got:      {got!r}')
    return ok

all_pass = True
results = []

print('=== IPv4 Network Properties ===')
n = Network('10.1.2.0/24')
results += [
    test('network_address', n.network_address, '10.1.2.0'),
    test('netmask', n.netmask, '255.255.255.0'),
    test('broadcast_address', n.broadcast_address, '10.1.2.255'),
    test('prefix_length', n.prefix_length, 24),
    test('num_hosts', n.num_hosts, 254),
    test('host_range[0]', n.host_range[0], '10.1.2.1'),
    test('host_range[1]', n.host_range[1], '10.1.2.254'),
    test('is_private (10.x)', n.is_private, True),
]

print('\n=== Containment ===')
results += [
    test('contains host', n.contains('10.1.2.100'), True),
    test('contains network addr', n.contains('10.1.2.0'), True),
    test('contains broadcast', n.contains('10.1.2.255'), True),
    test('does not contain outside', n.contains('10.1.3.1'), False),
]

print('\n=== Overlap Detection ===')
n1 = Network('10.0.0.0/8')
n2 = Network('10.1.0.0/16')
n3 = Network('192.168.0.0/16')
results += [
    test('overlaps (parent/child)', n1.overlaps(n2), True),
    test('overlaps (child/parent)', n2.overlaps(n1), True),
    test('no overlap (different ranges)', n1.overlaps(n3), False),
]

print('\n=== Supernet ===')
n24 = Network('10.1.2.0/24')
results += [
    test('supernet of /24', str(n24.supernet()), '10.1.0.0/23'),
    test('supernet of /23', str(n24.supernet().supernet()), '10.1.0.0/22'),
]

print('\n=== Subnet Splitting ===')
parent = Network('10.0.0.0/24')
subnets = parent.subnets(26)
results += [
    test('number of /26 subnets from /24', len(subnets), 4),
    test('first /26 subnet', str(subnets[0]), '10.0.0.0/26'),
    test('last /26 subnet', str(subnets[-1]), '10.0.0.192/26'),
]

print('\n=== Private Address Detection ===')
results += [
    test('10.0.0.0/8 is private', Network('10.0.0.0/8').is_private, True),
    test('172.16.0.0/12 is private', Network('172.16.0.0/12').is_private, True),
    test('192.168.0.0/16 is private', Network('192.168.0.0/16').is_private, True),
    test('8.8.8.0/24 is not private', Network('8.8.8.0/24').is_private, False),
]

print('\n=== find_overlaps ===')
overlapping = ['10.0.0.0/8', '10.1.0.0/16', '192.168.0.0/16']
pairs = Network.find_overlaps(overlapping)
results += [
    test('number of overlapping pairs', len(pairs), 1),
]

print('\n=== IPv6 Tests ===')
n6 = Network('2001:db8::/32')
results += [
    test('IPv6 network_address', n6.network_address, '2001:db8::'),
    test('IPv6 prefix_length', n6.prefix_length, 32),
    test('IPv6 contains', n6.contains('2001:db8::1'), True),
    test('IPv6 not contains', n6.contains('2001:db9::1'), False),
    test('fc00::/7 is private', Network('fc00::/7').is_private, True),
    test('2001:db8::/32 not private', n6.is_private, False),
]

all_pass = all(results)
passed = sum(results)
print(f'\n{passed}/{len(results)} tests passed')
if not all_pass:
    print('Fix failing tests before moving to Part 2')
else:
    print('All tests passed! Ready for Part 2.')


### Reflection 1.A

Describe the algorithm you implemented for `Network.aggregate()`. What is its time complexity?
How does it scale with the number of input prefixes?

**Your answer:**

*(Write here)*

### Reflection 1.B

Longest-prefix match requires checking every route in the table — O(n) per lookup.
Production routers handle millions of routes and need sub-microsecond lookups.
What data structures do they use, and why is linear scan not sufficient?

**Your answer:**

*(Write here)*

---
## Part 2: VLSM Allocator and Network Design

### Cell 2.1 — VLSM Allocator

Implement `VLSMAllocator` that allocates subnets from a parent network.
Sort requirements largest-first, allocate contiguously from the base address.

In [ ]:
# Cell 2.1 — VLSM allocator

class AllocationError(Exception):
    pass


class VLSMAllocator:
    """
    Variable Length Subnet Mask allocator.
    Allocates subnets of optimal size from a parent network.
    """

    def __init__(self, parent_cidr: str):
        self.parent = Network(parent_cidr)
        self._bits = self.parent._bits

    def allocate(self, host_requirements: list) -> list:
        """
        Allocate subnets for the given list of host requirements.

        Args:
            host_requirements: list of ints, number of hosts needed per subnet

        Returns:
            list of (Network, required_hosts) tuples in allocation order

        Raises:
            AllocationError if parent network is too small
        """
        # Sort requirements largest-first (pair with original index for traceability)
        indexed = sorted(enumerate(host_requirements), key=lambda x: x[1], reverse=True)

        parent_end = (self.parent._network_int +
                      (1 << (self._bits - self.parent.prefix_length)))
        next_addr = self.parent._network_int
        allocations = [None] * len(host_requirements)

        for orig_idx, n_hosts in indexed:
            prefix_len = hosts_to_prefix(n_hosts, self._bits)
            if prefix_len > self._bits:
                raise AllocationError(f'Cannot fit {n_hosts} hosts in a /{self._bits} subnet')

            # Align next_addr to the subnet boundary
            subnet_size = 1 << (self._bits - prefix_len)
            # Round up to alignment
            if next_addr % subnet_size != 0:
                next_addr += subnet_size - (next_addr % subnet_size)

            if next_addr + subnet_size > parent_end:
                raise AllocationError(
                    f'Parent network {self.parent} too small to allocate {n_hosts} hosts '
                    f'(ran out of space at {self.parent._int_to_str(next_addr)})'
                )

            addr_str = self.parent._int_to_str(next_addr)
            subnet = Network(f'{addr_str}/{prefix_len}')
            allocations[orig_idx] = (subnet, n_hosts)
            next_addr += subnet_size

        return allocations

    def report(self, allocations: list):
        """Print a formatted allocation table."""
        total_allocated = 0
        total_wasted = 0

        print(f"{'Subnet':<22} {'Prefix':>7} {'Required':>10} "
              f"{'Usable':>8} {'Waste':>7} {'Host Range'}")
        print('-' * 90)

        for subnet, required in allocations:
            usable = subnet.num_hosts
            waste = usable - required
            total_allocated += usable
            total_wasted += waste
            hr = subnet.host_range
            print(f"{subnet.network_address:<22} /{subnet.prefix_length:<6} "
                  f"{required:>10} {usable:>8} {waste:>7}  "
                  f"{hr[0]} – {hr[1]}")

        parent_hosts = self.parent.num_hosts
        utilization = 100 * total_allocated / parent_hosts if parent_hosts else 0
        print('-' * 90)
        print(f"Total allocated: {total_allocated}  "
              f"Wasted: {total_wasted}  "
              f"Parent capacity: {parent_hosts}  "
              f"Utilization: {utilization:.1f}%")


# Quick test
alloc = VLSMAllocator('10.0.0.0/24')
result = alloc.allocate([100, 50, 25, 10])
alloc.report(result)


### Cell 2.2 — Department Network Design

Design an addressing scheme for the university department using `172.16.32.0/20`.
Allocate all segments listed in the lab guide. Identify the largest remaining
contiguous block for future expansion.

In [ ]:
# Cell 2.2 — Department network design

DEPARTMENT_NETWORK = '172.16.32.0/20'

# Segment requirements — (name, hosts_needed)
SEGMENTS = [
    ('Faculty network',      200),
    ('Student lab A',        120),
    ('Student lab B',        120),
    ('Staff network',         60),
    ('Server network',        30),
    ('Management network',    14),
    ('Router link A-B',        2),
    ('Router link B-C',        2),
    ('Router link C-A',        2),
]

alloc = VLSMAllocator(DEPARTMENT_NETWORK)
requirements = [s[1] for s in SEGMENTS]
allocations = alloc.allocate(requirements)

print(f'Department Network: {DEPARTMENT_NETWORK}')
print(f'Total available hosts: {Network(DEPARTMENT_NETWORK).num_hosts}')
print()

# Display with segment names
print(f"{'Segment':<24} {'Subnet':<20} {'Prefix':>7} {'Req':>5} "
      f"{'Usable':>7} {'Waste':>6}")
print('-' * 80)

last_end = Network(DEPARTMENT_NETWORK)._network_int
for (name, req), (subnet, _) in zip(SEGMENTS, allocations):
    usable = subnet.num_hosts
    waste = usable - req
    print(f"{name:<24} {subnet.network_address:<20} /{subnet.prefix_length:<6} "
          f"{req:>5} {usable:>7} {waste:>6}")
    subnet_end = subnet._network_int + (1 << (32 - subnet.prefix_length))
    last_end = max(last_end, subnet_end)

# Identify future expansion block
parent = Network(DEPARTMENT_NETWORK)
parent_end = parent._network_int + (1 << (32 - parent.prefix_length))
remaining = parent_end - last_end

print()
print(f'Future expansion block starts at: {int_to_ip(last_end)}')
print(f'Remaining addresses: {remaining}')
if remaining > 0:
    # Find the largest aligned prefix that fits in the remaining space
    bits = int(math.log2(remaining)) if remaining > 0 else 0
    expansion_prefix = 32 - bits
    expansion_addr = int_to_ip(last_end)
    print(f'Largest aligned expansion block: {expansion_addr}/{expansion_prefix}')


### Cell 2.3 — Overlap Detection

Find all overlapping pairs in the provided configuration.
For each overlap, explain the routing conflict it would cause.

In [ ]:
# Cell 2.3 — Overlap detection

EXISTING_CONFIG = [
    '10.0.0.0/8',
    '10.1.0.0/16',
    '10.1.2.0/24',
    '192.168.0.0/16',
    '192.168.1.0/24',
    '192.168.1.128/25',
    '172.16.0.0/12',
    '172.16.1.0/24',
    '10.2.0.0/15',
]

overlaps = Network.find_overlaps(EXISTING_CONFIG)

print(f'Configuration has {len(EXISTING_CONFIG)} prefixes')
print(f'Overlapping pairs found: {len(overlaps)}')
print()

for net1, net2 in sorted(overlaps, key=lambda p: str(p[0])):
    # Determine which is more specific
    if net1.prefix_length <= net2.prefix_length:
        less, more = net1, net2
    else:
        less, more = net2, net1
    print(f'  {less} (/{less.prefix_length}) overlaps {more} (/{more.prefix_length})')
    print(f'  → {less} covers {more}')
    print()


**Overlap explanations:**

For each overlapping pair found above, describe the routing conflict it would cause:

*(Write here — one paragraph per pair)*

### Cell 2.4 — IPv6 Dual-Stack Transition

Design a dual-stack addressing plan mapping your IPv4 `/24` subnets to IPv6 `/64` subnets
within `2001:db8:1234::/48`.

In [ ]:
# Cell 2.4 — IPv6 dual-stack transition

IPV4_SPACE = '10.0.0.0/8'
IPV6_ALLOC = '2001:db8:1234::/48'

# How many /64 subnets does a /48 provide?
v6_parent = Network(IPV6_ALLOC)
slash64_count = 2 ** (64 - 48)
print(f'IPv6 allocation: {IPV6_ALLOC}')
print(f'Number of /64 subnets in a /48: {slash64_count:,}')
print(f'  = 2^(64-48) = 2^16 = {slash64_count}')

# How does this compare to total IPv4 host count in 10.0.0.0/8?
v4_parent = Network(IPV4_SPACE)
v4_hosts = v4_parent.num_hosts
print(f'\nIPv4 space: {IPV4_SPACE}')
print(f'Total IPv4 hosts in {IPV4_SPACE}: {v4_hosts:,}')
print(f'IPv6 /64 subnets available: {slash64_count:,}')
print(f'Each /64 provides {2**64:,} addresses')
print(f'Ratio of IPv6/64 subnets to IPv4 hosts: {slash64_count/v4_hosts:.4f}')

# Design the mapping
# Show first few IPv4 /24s and their corresponding IPv6 /64s
print('\nSample dual-stack mapping (first 5 subnets):')
print(f"{'IPv4 /24':<22} {'IPv6 /64'}")
print('-' * 60)

# Get first 5 /24 subnets of 10.0.0.0/8
v4_slash24s = v4_parent.subnets(24)
v6_slash64s = v6_parent.subnets(64)

for v4, v6 in zip(v4_slash24s[:5], v6_slash64s[:5]):
    print(f'{str(v4):<22} {str(v6)}')

print(f'...')
print(f'\nTotal IPv4 /24s available: {len(v4_slash24s):,}')
print(f'Total IPv6 /64s available: {slash64_count:,}')
print(f'Remaining IPv6 /64s after mapping all IPv4 /24s: {slash64_count - len(v4_slash24s):,}')


### Reflection 2.A

Your VLSM allocator sorts requirements largest-first. Why does this order produce a more
efficient allocation than smallest-first or arbitrary order? Construct a small example
that demonstrates the difference.

**Your answer:**

*(Write here — include the example)*

### Reflection 2.B

The overlap detection found several conflicts. In a real network, what symptoms would
a network administrator observe suggesting an IP overlap exists? How would they diagnose
it without knowing the configuration error in advance?

**Your answer:**

*(Write here)*

---
## Part 3: Routing Table Analysis

Load the provided routing table and analyze it using your library.

### Cell 3.1 — Load and Display

Load `lab6_routes.csv` into a `RoutingTable`. Show a summary and prefix length histogram.

In [ ]:
# Cell 3.1 — Load routing table

rt = RoutingTable()
route_metadata = []

with open(ROUTES_FILE) as f:
    reader = csv.DictReader(f)
    for row in reader:
        rt.add(row['prefix'], row['next_hop'],
               metric=int(row.get('metric', 0)),
               source=row.get('source', 'unknown'))
        route_metadata.append(row)

print(f'Routes loaded: {len(rt)}')

# Summarize by source protocol
sources = Counter(r.get('source', 'unknown') for r in route_metadata)
print('\nRoutes by source protocol:')
for src, count in sources.most_common():
    print(f'  {src:<15} {count}')

# Prefix length distribution
prefix_lens = [net.prefix_length for net, _, _ in rt.routes]

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(prefix_lens, bins=range(0, 33), edgecolor='white', align='left')
ax.set_xlabel('Prefix Length')
ax.set_ylabel('Number of Routes')
ax.set_title('Routing Table — Prefix Length Distribution')
ax.set_xticks(range(0, 33, 4))
plt.tight_layout()
plt.show()

print(f'\nPrefix length summary:')
for pl, count in sorted(Counter(prefix_lens).items()):
    print(f'  /{pl:<3} {count} routes')


### Cell 3.2 — Lookup Test

Look up all 15 test destinations. For each, show the matched prefix and next hop.

In [ ]:
# Cell 3.2 — Lookup test

test_destinations = [
    '10.1.2.5', '10.1.3.100', '10.2.0.1', '172.16.5.5',
    '192.168.10.50', '10.0.0.1', '8.8.8.8', '1.1.1.1',
    '10.255.255.255', '172.31.0.1', '192.0.2.1', '10.1.0.255',
    '172.16.0.1', '10.100.0.1', '203.0.113.5'
]

print(f"{'Destination':<18} {'Matched Prefix':<22} {'Next Hop':<18} {'Source'}")
print('-' * 75)

for dest in test_destinations:
    result = rt.lookup(dest)
    if result:
        net, nh, meta = result
        print(f"{dest:<18} {str(net):<22} {nh:<18} {meta.get('source', '')}")
    else:
        print(f"{dest:<18} {'(no match)':<22} {'—':<18}")


### Cell 3.3 — Covering Route Analysis

Find all covering route pairs. For each, explain what would happen if the
more-specific route were withdrawn.

In [ ]:
# Cell 3.3 — Covering routes

covering = rt.find_covering_routes()

print(f'Covering route pairs: {len(covering)}')
print()
print(f"{'Less specific (covering)':<25} {'Next hop':<18} {'More specific':<25} {'Next hop'}")
print('-' * 90)

for (less, nh_less), (more, nh_more) in sorted(covering, key=lambda p: str(p[0][0])):
    print(f"{str(less):<25} {nh_less:<18} {str(more):<25} {nh_more}")
print()


**Analysis — what happens when the more-specific route is withdrawn?**

For each covering pair above, explain the traffic engineering implication:

*(Write here — one paragraph per pair or group of pairs)*

### Cell 3.4 — Aggregation Opportunities

Find routes that could be aggregated. Show the before and after tables.

In [ ]:
# Cell 3.4 — Aggregation opportunities

# Group routes by next_hop
by_next_hop = defaultdict(list)
for net, nh, meta in rt.routes:
    by_next_hop[nh].append(net)

print('Routes grouped by next hop:')
aggregation_candidates = []

for nh, nets in sorted(by_next_hop.items()):
    prefixes = [str(n) for n in sorted(nets)]
    aggregated = Network.aggregate(prefixes)
    savings = len(nets) - len(aggregated) if aggregated else 0

    if savings > 0:
        aggregation_candidates.append((nh, nets, aggregated, savings))
        print(f'\n  Next hop: {nh}')
        print(f'  Before ({len(nets)} routes): {", ".join(prefixes)}')
        print(f'  After  ({len(aggregated)} routes): {", ".join(str(a) for a in aggregated)}')
        print(f'  Savings: {savings} routes')

total_savings = sum(c[3] for c in aggregation_candidates)
print(f'\nTotal routes eliminatable through aggregation: {total_savings}')
print(f'Current routes: {len(rt)}')
print(f'After aggregation: {len(rt) - total_savings}')


### Reflection 3.A

Cell 3.3 found cases where a specific route is covered by a less-specific one.
Why would an operator add a more-specific route when a covering route already exists?
Give a real-world example.

**Your answer:**

*(Write here)*

### Reflection 3.B

Your aggregation analysis found routes that could be combined. In practice, what prevents
network operators from always aggregating? What information is lost when routes are aggregated?

**Your answer:**

*(Write here)*

---
## Part 4: BGP Prefix Analysis

Analyze a real BGP routing table excerpt using your Network library.

### Cell 4.1 — Prefix Length Distribution

Plot the prefix length distribution. Compare to the known internet BGP table distribution.

In [ ]:
# Cell 4.1 — Load BGP data and plot prefix length distribution

with open(BGP_FILE) as f:
    bgp_data = json.load(f)

print(f'BGP entries loaded: {len(bgp_data)}')

# Parse all prefixes
bgp_networks = []
for entry in bgp_data:
    try:
        bgp_networks.append({
            'network':   Network(entry['prefix']),
            'origin_as': entry.get('origin_as', ''),
            'as_path':   entry.get('as_path', []),
            'next_hop':  entry.get('next_hop', ''),
        })
    except Exception as e:
        print(f'  Skipping {entry.get("prefix")}: {e}')

print(f'Valid BGP prefixes: {len(bgp_networks)}')

prefix_lens = [e['network'].prefix_length for e in bgp_networks]
pl_counts = Counter(prefix_lens)

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(pl_counts.keys(), pl_counts.values(), color='steelblue', edgecolor='white')
ax.set_xlabel('Prefix Length')
ax.set_ylabel('Number of BGP Prefixes')
ax.set_title('BGP Table Excerpt — Prefix Length Distribution')
ax.set_xticks(range(0, 33, 2))
plt.tight_layout()
plt.show()

print('\nPrefix length distribution:')
for pl in sorted(pl_counts.keys()):
    bar = '█' * pl_counts[pl]
    print(f'  /{pl:<3} {pl_counts[pl]:>5}  {bar[:60]}')

slash24_pct = 100 * pl_counts.get(24, 0) / len(bgp_networks)
print(f'\n/24 prefixes: {pl_counts.get(24, 0)} ({slash24_pct:.1f}% of total)')


### Cell 4.2 — AS Path Analysis

Compute AS path length distribution. Find and discuss the longest AS path.

In [ ]:
# Cell 4.2 — AS path analysis

path_lengths = [len(e['as_path']) for e in bgp_networks if e['as_path']]

print(f'AS path length statistics:')
print(f'  Mean:   {np.mean(path_lengths):.2f}')
print(f'  Median: {np.median(path_lengths):.1f}')
print(f'  Max:    {max(path_lengths)}')
print(f'  Min:    {min(path_lengths)}')

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(path_lengths, bins=range(1, max(path_lengths) + 2),
        edgecolor='white', align='left')
ax.set_xlabel('AS Path Length (hops)')
ax.set_ylabel('Number of Prefixes')
ax.set_title('BGP AS Path Length Distribution')
plt.tight_layout()
plt.show()

# Longest path
longest = max(bgp_networks, key=lambda e: len(e['as_path']))
print(f'\nLongest AS path: {len(longest["as_path"])} hops')
print(f'  Prefix: {longest["network"]}')
print(f'  AS path: {" → ".join(str(a) for a in longest["as_path"])}')


### Cell 4.3 — Deaggregation Detection

Find prefixes where a more-specific prefix coexists with a covering prefix from the same
origin AS. Show the pairs and explain the traffic engineering motivation.

In [ ]:
# Cell 4.3 — Deaggregation detection

# Sort by network address then prefix length
sorted_bgp = sorted(bgp_networks, key=lambda e: (e['network']._network_int,
                                                   e['network'].prefix_length))

deaggregated_pairs = []

for i, entry_i in enumerate(sorted_bgp):
    for j, entry_j in enumerate(sorted_bgp):
        if i == j:
            continue
        net_i = entry_i['network']
        net_j = entry_j['network']

        # net_i is the covering prefix (less specific)
        # net_j is the deaggregated prefix (more specific)
        if (net_i.prefix_length < net_j.prefix_length and
                net_i.contains(net_j.network_address) and
                entry_i['origin_as'] == entry_j['origin_as']):
            deaggregated_pairs.append((entry_i, entry_j))

# Deduplicate (same covering prefix, multiple specifics)
seen = set()
unique_pairs = []
for covering, specific in deaggregated_pairs:
    key = (str(covering['network']), str(specific['network']))
    if key not in seen:
        seen.add(key)
        unique_pairs.append((covering, specific))

print(f'Deaggregated prefix pairs found: {len(unique_pairs)}')
print()

for covering, specific in unique_pairs[:10]:   # show first 10
    print(f'  Covering:  {covering["network"]}  AS{covering["origin_as"]}')
    print(f'             path: {"->".join(str(a) for a in covering["as_path"])}')
    print(f'  Specific:  {specific["network"]}  AS{specific["origin_as"]}')
    print(f'             path: {"->".join(str(a) for a in specific["as_path"])}')
    print()

if len(unique_pairs) > 10:
    print(f'  ... and {len(unique_pairs) - 10} more pairs')


**Deaggregation explanation:**

Why would an operator advertise both a covering prefix and a more-specific subset?

*(Write here — explain the traffic engineering motivation)*

### Cell 4.4 — Address Space Coverage

Compute what fraction of total IPv4 space is covered by this BGP excerpt.
Which /8 blocks have the most coverage?

In [ ]:
# Cell 4.4 — Address space coverage

total_ipv4 = 2**32   # total addresses in IPv4 space

# Sum up addresses covered by all BGP prefixes
# (This double-counts overlapping prefixes — for a rough estimate this is acceptable)
covered_addrs = sum(
    2 ** (32 - e['network'].prefix_length)
    for e in bgp_networks
    if not e['network']._is_v6
)

coverage_pct = 100 * covered_addrs / total_ipv4
print(f'Approximate address space coverage: {coverage_pct:.2f}%')
print(f'  ({covered_addrs:,} of {total_ipv4:,} addresses)')
print('  Note: may overcount due to deaggregated prefixes')

# Which /8 blocks have the most prefixes?
slash8_counts = Counter()
for e in bgp_networks:
    if not e['network']._is_v6:
        # Get the /8 block this prefix belongs to
        first_octet = (e['network']._network_int >> 24) & 0xFF
        slash8_counts[first_octet] += 1

print(f'\nTop 10 /8 blocks by prefix count:')
print(f"{'Block':<12} {'Prefixes':>10}")
print('-' * 24)
for octet, count in slash8_counts.most_common(10):
    print(f"{octet}.0.0.0/8   {count:>10}")

# Plot /8 coverage
fig, ax = plt.subplots(figsize=(14, 4))
octets = list(slash8_counts.keys())
counts = [slash8_counts[o] for o in octets]
ax.bar(octets, counts, width=1, edgecolor='none')
ax.set_xlabel('/8 Block (first octet)')
ax.set_ylabel('Number of BGP Prefixes')
ax.set_title('BGP Prefix Distribution by /8 Block')
ax.set_xlim(0, 255)
plt.tight_layout()
plt.show()


### Reflection 4.A

Your deaggregation analysis found prefixes where an operator advertises both a covering
block and a more-specific subset. Explain the traffic engineering motivation.
What does the more-specific prefix allow the operator to control?

**Your answer:**

*(Write here)*

### Reflection 4.B

The BGP table is dominated by /24 prefixes. Many ISPs filter routes longer than /24.
What is the justification for this policy? What problem does it prevent?

**Your answer:**

*(Write here)*

---
## Submission Checklist

Before your final push, complete this checklist. Replace each `[ ]` with `[x]` when done.

- [ ] Cell 1.1: All core integer operation tests pass
- [ ] Cell 1.2: Network class — properties and methods working for IPv4 and IPv6
- [ ] Cell 1.3: RoutingTable — all lookup tests pass
- [ ] Cell 1.4: Full test suite — all tests pass including IPv6
- [ ] Reflections 1.A and 1.B completed
- [ ] Cell 2.1: VLSMAllocator works on test case
- [ ] Cell 2.2: Department allocation table shown with future expansion block identified
- [ ] Cell 2.3: All overlapping pairs found and explained
- [ ] Cell 2.4: IPv6 plan shown, subnet count answered
- [ ] Reflections 2.A and 2.B completed
- [ ] Cell 3.1: Routes loaded, histogram shown
- [ ] Cell 3.2: All 15 lookup results shown
- [ ] Cell 3.3: Covering pairs found and explained
- [ ] Cell 3.4: Aggregation opportunities shown with before/after
- [ ] Reflections 3.A and 3.B completed
- [ ] Cell 4.1: Prefix length histogram shown
- [ ] Cell 4.2: AS path distribution shown, longest path discussed
- [ ] Cell 4.3: Deaggregated pairs found with explanation
- [ ] Cell 4.4: Coverage percentage and /8 distribution shown
- [ ] Reflections 4.A and 4.B completed
- [ ] Notebook runs cleanly with Kernel → Restart Kernel and Run All Cells
- [ ] At least 5 commits with descriptive messages

Final commit message must be exactly: `Lab6 final submission`

---
*CMSC395 Computer Networks — Lab 6*  
*Push this notebook to your repository with the message: `Lab6 final submission`*